# Полный прогон пайплайна в Colab

Ноутбук только запускает скрипты из репозитория, вся логика лежит в `scripts/` и `src/`.

1. `Runtime → Change runtime type → T4 GPU`.
2. Один раз положить на Google Drive в папку `MyDrive/text-classification/` файл `kaggle.json` (Kaggle → Settings → API → Create New Token).
3. Если репозиторий приватный, добавить в Colab Secrets (значок ключа слева) `GITHUB_TOKEN` с правами `repo`.
4. `Runtime → Run all`.

Данные, эмбеддинги, веса моделей, логи TensorBoard и метрики сохраняются на Google Drive, поэтому не пропадут при отключении сессии.

In [ ]:
REPO = "semenovfilipp/text-classification"
BRANCH = "feature/text-classification"
DRIVE_DIR = "/content/drive/MyDrive/text-classification"
RUN_CLASSIC = True
RUN_EMBEDDINGS = True
RUN_EMBEDDINGS_GPU = True
RUN_FINETUNE = ["configs/finetune.yaml", "configs/finetune_bertweet.yaml"]
PUSH_RESULTS = True

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
auth = f"{token}@" if token else ""
os.chdir("/content")
if not os.path.exists("/content/text-classification"):
    !git clone -b {BRANCH} https://{auth}github.com/{REPO}.git /content/text-classification
os.chdir("/content/text-classification")
!git pull -q origin {BRANCH}
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt
!python -c "import nltk; [nltk.download(p, quiet=True) for p in ('stopwords', 'wordnet', 'omw-1.4')]" 

Папки `data`, `models`, `logs`, `results` в репозитории заменяются ссылками на Google Drive.

In [ ]:
import shutil
for d in ("data", "models", "logs", "results"):
    target = f"{DRIVE_DIR}/{d}"
    os.makedirs(target, exist_ok=True)
    if os.path.islink(d):
        continue
    if os.path.isdir(d):
        for name in os.listdir(d):
            dst = os.path.join(target, name)
            if not os.path.exists(dst):
                shutil.move(os.path.join(d, name), dst)
        shutil.rmtree(d)
    os.symlink(target, d)
!ls -la | grep -- "->" 

### Данные с Kaggle

In [ ]:
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
for f in ("kaggle.json", "access_token"):
    if os.path.exists(f"{DRIVE_DIR}/{f}"):
        shutil.copy(f"{DRIVE_DIR}/{f}", os.path.expanduser(f"~/.kaggle/{f}"))
        os.chmod(os.path.expanduser(f"~/.kaggle/{f}"), 0o600)
if not os.path.exists("data/raw/train.csv"):
    !kaggle competitions download -c nlp-getting-started -p data/raw && unzip -o -q data/raw/nlp-getting-started.zip -d data/raw && rm data/raw/nlp-getting-started.zip
if not os.path.exists("data/external/glove-twitter-100.gz"):
    !mkdir -p data/external && wget -q -O data/external/glove-twitter-100.gz https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-twitter-100/glove-twitter-100.gz
!ls -la data/raw data/external

### Предобработка: сплит `v1` и версии датасета для классики

In [ ]:
!python scripts/preprocess.py --config configs/classic.yaml

### 1. Классический NLP

In [ ]:
if RUN_CLASSIC:
    !python scripts/train_classic.py --config configs/classic.yaml 2>&1 | tee logs/classic.log | grep -E "f1=|best|model:|submission:|Error" 

### 2. Эмбеддинги трансформеров + LogReg/CatBoost

In [ ]:
if RUN_EMBEDDINGS:
    !python scripts/train_embeddings.py --config configs/embeddings.yaml 2>&1 | tee logs/embeddings.log | grep -E "f1=|best|model:|submission:|Error"
if RUN_EMBEDDINGS_GPU:
    !python scripts/train_embeddings.py --config configs/embeddings_gpu.yaml 2>&1 | tee logs/embeddings_gpu.log | grep -E "f1=|best|model:|submission:|Error" 

### 3. Fine-tune трансформера

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

In [ ]:
for cfg in RUN_FINETUNE:
    name = os.path.basename(cfg).replace(".yaml", "")
    !python scripts/finetune.py --config {cfg} 2>&1 | tee logs/{name}.log | grep -E "eval_f1|f1=|model:|submission:|Error" 

### Итоговая таблица

In [ ]:
!python scripts/evaluate.py --top 15

In [ ]:
import pandas as pd
df = pd.read_csv("results/metrics.csv").drop_duplicates("run", keep="last")
df.sort_values("f1", ascending=False).groupby("approach").head(3)[["approach", "run", "f1", "accuracy", "roc_auc", "train_time_s", "infer_time_s"]]

### Отправка метрик в ветку

В git попадают только `results/metrics.csv` и `results/metrics_table.md`. Данные и веса остаются на Drive.

In [ ]:
if PUSH_RESULTS and token:
    os.makedirs("/content/results_commit", exist_ok=True)
    !git config user.email "colab@users.noreply.github.com" && git config user.name "colab-runner"
    !rm results && mkdir results && cp {DRIVE_DIR}/results/metrics.csv {DRIVE_DIR}/results/metrics_table.md results/
    !git add results/metrics.csv results/metrics_table.md && git commit -m "Add GPU run metrics from Colab" && git push origin {BRANCH}
    !rm -rf results && ln -s {DRIVE_DIR}/results results
elif PUSH_RESULTS:
    print("GITHUB_TOKEN не задан: скачайте metrics.csv и metrics_table.md из", f"{DRIVE_DIR}/results")